# Email Support Model — Fine-tuning on Google Colab

**Base model:** `meta-llama/Llama-3.2-3B-Instruct` (same as production `llama3.2:3b`)
**Method:** QLoRA 4-bit — trains on free T4 GPU in ~20-30 minutes
**Output:** GGUF Q4_K_M — loads directly into Ollama

### Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `combined_train.jsonl` from your machine (generated by `prepare_dataset.py`)

### What you need from your machine:
```bash
cd "/home/parthiv/web/Email Automation"
source backend/venv/bin/activate
python3 finetune/prepare_dataset.py
# Upload: data/training/combined_train.jsonl
```

In [ ]:
# Verify GPU is available
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print('GPU:', r.stdout.strip())
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU')

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade trl peft accelerate bitsandbytes datasets transformers
print('Done.')

## Step 2 — Upload Dataset

In [ ]:
from google.colab import files
print('Upload combined_train.jsonl from your machine:')
uploaded = files.upload()
assert 'combined_train.jsonl' in uploaded, 'ERROR: File must be named combined_train.jsonl'
print(f'Uploaded {len(uploaded["combined_train.jsonl"])/1024:.1f} KB')

## Step 3 — Load Model (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    # Same base as production llama3.2:3b — fine-tune becomes a direct replacement
    model_name     = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Step 4 — Apply LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()

## Step 5 — Prepare Dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files='combined_train.jsonl', split='train')
print(f'Records loaded: {len(raw)}')

# Format into chat template — matches the exact prompt format used in backend/generator.py
def format_record(row):
    messages = [
        {"role": "system",    "content": row['instruction']},
        {"role": "user",      "content": row['input']},
        {"role": "assistant", "content": row['output']},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = raw.map(format_record, remove_columns=raw.column_names)

print('\nSample formatted record:')
print(dataset[0]['text'][:400])
print('...')

## Step 6 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_seq_length              = MAX_SEQ_LENGTH,
        dataset_num_proc            = 2,
        packing                     = True,   # required when padding_free=True
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 5,
        warmup_steps                = 10,
        learning_rate               = 2e-4,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        output_dir                  = '/content/checkpoints',
        report_to                   = 'none',
    ),
)

print('Starting training...')
print('Expected time: 20-30 minutes on T4 GPU\n')
stats = trainer.train()
print(f'\nFinal loss: {stats.training_loss:.4f}')

## Step 7 — Save as GGUF (for Ollama)

In [ ]:
# Save merged model as GGUF Q4_K_M — best balance of quality and CPU speed
print('Saving GGUF model (this takes 5-10 minutes)...')

model.save_pretrained_gguf(
    '/content/email-support-gguf',
    tokenizer,
    quantization_method = 'q4_k_m'
)

import os
gguf_files = [f for f in os.listdir('/content/email-support-gguf') if f.endswith('.gguf')]
print(f'\nGGUF files created: {gguf_files}')
for f in gguf_files:
    size = os.path.getsize(f'/content/email-support-gguf/{f}') / 1e9
    print(f'  {f}: {size:.2f} GB')

## Step 8 — Download GGUF to your machine

In [ ]:
from google.colab import files
import os, glob

# Unsloth appends _gguf to the output dir — search broadly
gguf_files = glob.glob('/content/**/*.gguf', recursive=True)
print('Found GGUF files:', gguf_files)

if gguf_files:
    # Prefer Q4_K_M if multiple exist
    gguf_path = next((f for f in gguf_files if 'Q4_K_M' in f), gguf_files[0])
    print(f'\nDownloading: {gguf_path}')
    print(f'Size: {os.path.getsize(gguf_path)/1e9:.2f} GB')
    files.download(gguf_path)
    print('\nDownload started! Save it to your Downloads folder.')
else:
    print('No GGUF file found anywhere under /content/')

## Step 9 — Load into Ollama (run on your Ubuntu machine)

After the GGUF file downloads, run these commands on your machine:

```bash
# 1. Move the downloaded GGUF to your project
mv ~/Downloads/*.gguf "/home/parthiv/web/Email Automation/data/finetuned/email-support-v2.gguf"

# 2. Create Modelfile
cat > "/home/parthiv/web/Email Automation/data/finetuned/Modelfile-v2" << 'EOF'
FROM /home/parthiv/web/Email Automation/data/finetuned/email-support-v2.gguf

SYSTEM """You are a professional customer support agent.
Analyze customer emails and respond with ONLY valid JSON.
Be professional, concise, and helpful."""

PARAMETER temperature 0.2
PARAMETER num_ctx 2048
EOF

# 3. Load into Ollama
ollama create email-support-v2 -f "/home/parthiv/web/Email Automation/data/finetuned/Modelfile-v2"

# 4. Test it
ollama run email-support-v2 "Reply to: I was charged twice this month."

# 5. Update .env to use fine-tuned model
# Change: LLM_MODEL=email-support-v2
```
